# StockSense AI Model Training

This notebook trains a hybrid inventory model. The deterministic inventory policy calculates the
minimum safe reorder amount, while the machine learning model learns an adjustment from messy
historical ordering behavior.



In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SCRIPT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent if SCRIPT_DIR.name == "model" else SCRIPT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.ml_features import (  # noqa: E402
    BASE_FEATURES,
    CATEGORICAL_FEATURES,
    ENGINEERED_FEATURES,
    NUMERIC_FEATURES,
    InventoryFeatureEngineer,
    add_inventory_features,
)

RANDOM_STATE = 42
DATA_PATH = SCRIPT_DIR / "restaurant_inventory_with_targets.csv"
MODEL_PATH = SCRIPT_DIR / "reorder_model_tuned.pkl"



## Load Data



In [2]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print("Dataset loaded successfully")
print("Shape:", df.shape)
print(df.head())



Dataset loaded successfully
Shape: (1000, 16)
         Date  Item_ID Item_Name Category Subcategory Unit  Current_Stock  \
0  2025-06-10        1    Paneer      Veg       Dairy   kg          21.45   
1  2025-06-10        2    Tomato      Veg   Vegetable   kg          12.84   
2  2025-06-10        3     Onion      Veg   Vegetable   kg          22.35   
3  2025-06-10        4   Chicken  Non-Veg        Meat   kg           8.36   
4  2025-06-10        5    Mutton  Non-Veg        Meat   kg          12.31   

   Reorder_Level  Daily_Usage  Lead_Time  Price_per_Unit Supplier_Name  \
0           8.12         2.19          1             450    Supplier A   
1           5.34         0.95          4              40    Supplier A   
2           4.49         4.86          4              35    Supplier A   
3           3.16         3.25          3             250    Supplier C   
4           6.19         1.81          3             600    Supplier A   

   Seasonal_Factor  Waste_Percentage  Waste_Ad

## Clean Data And Build Inventory Policy Features



In [3]:
target = "Inventory_To_Order"
required_columns = BASE_FEATURES + [target]
missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

for column in NUMERIC_FEATURES + [target]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna(subset=required_columns).copy()
df = add_inventory_features(df)

print("Cleaned dataset shape:", df.shape)
print(df[BASE_FEATURES + ["Waste_Adjusted_Minimum", target]].head())



Cleaned dataset shape: (1000, 23)
  Item_Name Category Subcategory Unit  Current_Stock  Reorder_Level  \
0    Paneer      Veg       Dairy   kg          21.45           8.12   
1    Tomato      Veg   Vegetable   kg          12.84           5.34   
2     Onion      Veg   Vegetable   kg          22.35           4.49   
3   Chicken  Non-Veg        Meat   kg           8.36           3.16   
4    Mutton  Non-Veg        Meat   kg          12.31           6.19   

   Daily_Usage  Lead_Time  Price_per_Unit  Seasonal_Factor  Waste_Percentage  \
0         2.19          1             450             1.11              2.98   
1         0.95          4              40             0.81              3.54   
2         4.86          4              35             1.23              4.96   
3         3.25          3             250             0.90              3.06   
4         1.81          3             600             1.07              3.09   

   Waste_Adjusted_Minimum  Inventory_To_Order  
0         

## Simulate Controlled Real-World Label Noise

The project intentionally keeps noisy labels to mimic messy real restaurant ordering behavior.
The noise is seeded for reproducibility and clipped so the target never becomes physically impossible.



In [4]:
rng = np.random.default_rng(RANDOM_STATE)
noise_scale = min(1.0, df[target].std() * 0.05)

df["Raw_Target"] = df[target]
df["Noisy_Target"] = (
    df[target] + rng.normal(loc=0, scale=noise_scale, size=len(df))
).clip(lower=0)

df["Target_Adjustment"] = df["Noisy_Target"] - df["Waste_Adjusted_Minimum"]

print(f"Noise scale: {noise_scale:.2f}")
print("Raw target min:", round(df["Raw_Target"].min(), 2))
print("Noisy target min:", round(df["Noisy_Target"].min(), 2))
print("Adjustment target summary:")
print(df["Target_Adjustment"].describe().round(2))



Noise scale: 0.40
Raw target min: 0.45
Noisy target min: 0.0
Adjustment target summary:
count    1000.00
mean        7.05
std         5.49
min        -2.71
25%         3.24
50%         5.49
75%         9.94
max        34.31
Name: Target_Adjustment, dtype: float64


## Train/Test Split



In [5]:
X = df[BASE_FEATURES]
y_adjustment = df["Target_Adjustment"]
y_final = df["Noisy_Target"]

X_train, X_test, y_train, y_test, y_final_train, y_final_test = train_test_split(
    X,
    y_adjustment,
    y_final,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])



Training rows: 800
Testing rows: 200


## Model Candidates

Each model predicts the adjustment above the operational minimum. The final recommendation is
`max(operational minimum, operational minimum + model adjustment)`.



In [6]:
model_features = NUMERIC_FEATURES + ENGINEERED_FEATURES

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
        ("num", "passthrough", model_features),
    ],
    verbose_feature_names_out=False,
)

candidate_models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        l2_regularization=0.05,
        random_state=RANDOM_STATE,
    ),
}

def make_pipeline(regressor):
    return Pipeline(
        steps=[
            ("feature_engineering", InventoryFeatureEngineer()),
            ("preprocessor", preprocessor),
            ("regressor", regressor),
        ]
    )



## Evaluate Candidate Models



In [7]:
def operational_minimum_for(X_values):
    return add_inventory_features(X_values)["Waste_Adjusted_Minimum"].to_numpy()


def evaluate_model(name, pipeline):
    pipeline.fit(X_train, y_train)

    adjustment_pred = pipeline.predict(X_test)
    operational_minimum = operational_minimum_for(X_test)
    final_pred = np.maximum(operational_minimum, operational_minimum + adjustment_pred)
    final_pred = np.clip(final_pred, 0, None)

    return {
        "Model": name,
        "MAE": mean_absolute_error(y_final_test, final_pred),
        "RMSE": np.sqrt(mean_squared_error(y_final_test, final_pred)),
        "R2": r2_score(y_final_test, final_pred),
        "Adjustment_MAE": mean_absolute_error(y_test, adjustment_pred),
        "Pipeline": pipeline,
    }


results = []
for model_name, regressor in candidate_models.items():
    print(f"Training {model_name}...")
    results.append(evaluate_model(model_name, make_pipeline(regressor)))

comparison = pd.DataFrame(results).drop(columns=["Pipeline"]).sort_values("MAE")
comparison



Training Random Forest...
Training Extra Trees...
Training Hist Gradient Boosting...


,Model,MAE,RMSE,R2,Adjustment_MAE
2,Hist Gradient Boosting,0.722027,1.249678,0.979708,0.717699
1,Extra Trees,0.754386,1.363008,0.975861,0.752523
0,Random Forest,0.804996,1.386382,0.975026,0.801913


## Select Best Model And Save



In [8]:
best_result = min(results, key=lambda item: item["MAE"])
best_model = best_result["Pipeline"]

print("Best model:", best_result["Model"])
print(f"MAE: {best_result['MAE']:.2f}")
print(f"RMSE: {best_result['RMSE']:.2f}")
print(f"R2: {best_result['R2']:.3f}")

joblib.dump(best_model, MODEL_PATH)
print(f"Saved model to {MODEL_PATH}")



Best model: Hist Gradient Boosting
MAE: 0.72
RMSE: 1.25
R2: 0.980
Saved model to c:\Users\dell0\Documents\GitHub\StockSenseAI\model\reorder_model_tuned.pkl


## Edge Case Checks



In [9]:
edge_cases = pd.DataFrame(
    [
        {
            "Item_Name": "Eggs",
            "Category": "Non-Veg",
            "Subcategory": "Poultry",
            "Unit": "pieces",
            "Current_Stock": 20,
            "Reorder_Level": 70,
            "Daily_Usage": 4,
            "Lead_Time": 2,
            "Price_per_Unit": 6,
            "Seasonal_Factor": 0.87,
            "Waste_Percentage": 1.52,
        },
        {
            "Item_Name": "Eggs",
            "Category": "Non-Veg",
            "Subcategory": "Poultry",
            "Unit": "pieces",
            "Current_Stock": 23,
            "Reorder_Level": 24,
            "Daily_Usage": 20,
            "Lead_Time": 0,
            "Price_per_Unit": 0.2,
            "Seasonal_Factor": 0.87,
            "Waste_Percentage": 1.52,
        },
    ]
)

edge_operational_minimum = operational_minimum_for(edge_cases)
edge_adjustment = best_model.predict(edge_cases)
edge_final = np.maximum(edge_operational_minimum, edge_operational_minimum + edge_adjustment)

edge_results = edge_cases[["Item_Name", "Current_Stock", "Reorder_Level", "Daily_Usage", "Lead_Time"]].copy()
edge_results["Operational_Minimum"] = np.ceil(edge_operational_minimum).astype(int)
edge_results["Model_Adjustment"] = np.round(edge_adjustment, 2)
edge_results["Final_Recommendation"] = np.ceil(edge_final).astype(int)
edge_results



,Item_Name,Current_Stock,Reorder_Level,Daily_Usage,Lead_Time,Operational_Minimum,Model_Adjustment,Final_Recommendation
0,Eggs,20,70,4,2,59,1.41,61
1,Eggs,23,24,20,0,2,-0.29,2


## Load Saved Model Smoke Test



In [10]:
loaded_model = joblib.load(MODEL_PATH)
sample = X_test.iloc[[0]]
sample_operational_minimum = operational_minimum_for(sample)[0]
sample_adjustment = loaded_model.predict(sample)[0]
sample_final = max(sample_operational_minimum, sample_operational_minimum + sample_adjustment)

print("Sample input:")
print(sample)
print(f"Operational minimum: {sample_operational_minimum:.2f}")
print(f"Model adjustment: {sample_adjustment:.2f}")
print(f"Final recommendation: {sample_final:.2f}")


Sample input:
    Item_Name Category Subcategory Unit  Current_Stock  Reorder_Level  \
521    Tomato      Veg   Vegetable   kg          11.63           7.35   

     Daily_Usage  Lead_Time  Price_per_Unit  Seasonal_Factor  Waste_Percentage  
521          2.5          1              40             1.19              1.26  
Operational minimum: 0.00
Model adjustment: 3.00
Final recommendation: 3.00
